# Module 06 · Unit 03
# Decision Trees and Security Classifiers

| | |
|---|---|
| **Estimated time** | 60–70 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Access Control & Policy Logic \[AC\] · Adversarial Enumeration \[AE\] |
| **Refresher** | Module 06 · Unit 01–02 — Tree Definitions, Parse Trees |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Define a **decision tree** formally and distinguish it from a parse tree
2. Build a decision tree for a multi-attribute security access policy
3. Compute the **depth** required to express a given policy exactly
4. Identify the **root-to-leaf path** for any input and interpret it as a policy trace
5. Explain the **adversarial path-finding problem**: how an attacker probes a classifier
6. Connect decision tree depth to Boolean formula complexity and to the ABAC policies from Module 01


## 🔣 Symbol Primer

Decision trees reuse tree notation from Unit 01 with a few specialised additions.

| Symbol | Name | Read it as | Meaning |
|---|---|---|---|
| $f(v)$ | Node test | "test at $v$" | The Boolean predicate evaluated at internal node $v$ |
| $\text{label}(\ell)$ | Leaf label | "label of leaf $\ell$" | The classification assigned at leaf $\ell$ |
| $\text{path}(x)$ | Input path | "path of $x$" | Root-to-leaf path followed when input $x$ is classified |
| $\text{depth}(T)$ | Tree depth | "depth of tree $T$" | Height of $T$ — maximum length of any root-to-leaf path |
| $VC(\mathcal{H})$ | VC dimension | "VC dimension of $\mathcal{H}$" | Maximum number of points a hypothesis class can shatter |

> **The connection to Module 01.** A decision tree with Boolean tests at each
> internal node is a graphical representation of a Boolean formula — the same
> formulas from Module 01. Every internal node is a propositional variable;
> every root-to-leaf path is one row in the truth table; every leaf is a
> True/False outcome. The tree makes the evaluation path of any specific
> input visible, which truth tables do not.

---


## 1 · Decision Trees — Formal Definition

A **decision tree** for a classification problem over inputs $x \in \mathcal{X}$
with output labels $y \in \mathcal{Y}$ is a rooted binary tree where:

- Each **internal node** $v$ is labelled with a feature test $f(v) : \mathcal{X} \to \{0, 1\}$
- Each internal node has a **left child** (test fails: $f(v) = 0$) and a **right child** (test passes: $f(v) = 1$)
- Each **leaf** $\ell$ is labelled with a class $\text{label}(\ell) \in \mathcal{Y}$

**Evaluating the tree on input $x$:**

Start at the root. At each internal node $v$, evaluate $f(v)(x)$:
- If $0$ (False): follow the left child
- If $1$ (True): follow the right child

Continue until a leaf is reached. The leaf's label is the classification.

### Decision Tree as Truth Table

For $n$ Boolean features, a complete binary tree of depth $n$ has $2^n$ leaves —
one for every row of the truth table. Each root-to-leaf path corresponds to one
specific combination of feature values. The leaf labels are the output column.

**This is Module 01 revisited.** The truth table for the ABAC policy
$A \wedge T \wedge \neg S$ has 8 rows (3 variables, $2^3$). The decision tree
for this policy has depth 3 and 8 leaves — the path to each leaf traces one
row of the truth table.

### Depth and Complexity

A decision tree of depth $d$ can represent any Boolean function on at most $d$ variables.
For $n$ variables:

- **Minimum depth** to express any function: $n$ (complete tree)
- **Minimum depth** for a specific function: depends on the function — a tautology
  needs depth 0 (a single leaf); an XOR needs depth $n$

Shallow trees are fast to evaluate but can only represent simple policies.
Deep trees can represent complex policies but take longer to evaluate — and
give attackers more "branches" to probe.


## 2 · Security Decision Trees — Policy as a Tree

The ABAC policy $A \wedge T \wedge \neg S$ from Module 01 can be expressed
as a decision tree:

```
Root: Is A (admin)?
  No  → DENY  (leaf)
  Yes → Is T (time in window)?
          No  → DENY  (leaf)
          Yes → Is S (suspended)?
                  Yes → DENY  (leaf)
                  No  → ALLOW (leaf)
```

This tree has depth 3 and 4 leaves. Note: because the policy is a conjunction,
the tree can short-circuit early — if $A$ is False, we deny immediately without
checking $T$ or $S$. This is the **lazy evaluation** property of AND-trees:
each AND clause is tested in order from the root, and we stop at the first failure.

### Leaf Labels as Security Decisions

In a security context, the leaf labels are not just True/False — they are
**security actions**:

| Label | Meaning |
|---|---|
| `ALLOW` | Grant access; proceed normally |
| `DENY` | Reject access; return 403 |
| `ESCALATE` | Require additional verification (MFA challenge, CAPTCHA) |
| `LOG_AND_ALLOW` | Allow but create audit record |
| `QUARANTINE` | Allow with restricted permissions; monitor closely |

A decision tree with five leaf labels is a multi-class classifier —
it assigns one of five security dispositions to every input.

### The Ordering Problem

The order of tests in a decision tree affects **efficiency** (which tests are
evaluated before denying) and **security** (which conditions are checked first,
allowing shorter paths to ALLOW for certain inputs).

For a policy $A \wedge T \wedge \neg S$, the attacker's preferred path to ALLOW
is the root-to-ALLOW leaf path. If $A$ is tested last, an attacker who fails
$A$ has already shown their $T$ and $S$ status — information leakage.
Placing the most discriminating test at the root minimises both evaluation
cost and information leakage.


## 3 · Adversarial Path-Finding

An attacker interacting with a decision tree-based classifier faces a specific
problem: find an input $x$ such that $\text{label}(\text{path}(x)) = \text{ALLOW}$.

### The Oracle Attack Model

Suppose the attacker can submit inputs and observe the classification (allow/deny)
but cannot see the tree structure. This is the **black-box oracle** model:

$$\text{oracle}(x) = \text{label}(\text{path}(x))$$

In this model, the attacker's goal is to find the ALLOW leaf by submitting
queries. The tree structure determines how hard this is:

- **Depth $d$ tree:** at most $d$ queries needed to identify the path to any leaf
- **Each query** reveals one bit of information (pass/fail at one node)
- **Total queries** to find the ALLOW path: $O(d)$

This is a formal statement of why **shallow, simple decision trees are easily
reverse-engineered** by adversaries who can interact with the classifier.

### Adversarial Examples as Path-Finding

An **adversarial example** in a machine learning classifier is an input crafted
to follow a specific path to a desired leaf. In a decision tree:

$$x_{\text{adv}} = \text{input satisfying all tests on the path to ALLOW}$$

For a tree with Boolean tests, the adversarial example is exactly the conjunction
of the feature values along the ALLOW path — the same structure as a truth table
row. For a tree with continuous features (like most ML classifiers), finding an
adversarial example requires knowing the split thresholds at each node.

### Connection to Module 05

The adversarial path-finding problem is structurally identical to the attack
graph problem from Module 05:

| Attack graph | Decision tree |
|---|---|
| Entry point vertex | Root node |
| Target vertex | ALLOW leaf |
| Edge traversal | Satisfying a node test |
| Minimum-hop path | Minimum-test path to ALLOW |
| Dijkstra's algorithm | Binary search through the tree |

The attacker is navigating from root to ALLOW leaf. The defender wants to
maximise the minimum cost of this navigation — by making each node test
difficult to satisfy (high CVSS cost edge), not by obscuring the tree structure.


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[AC\] \[AE\]

| Decision tree concept | Security meaning |
|---|---|
| **Internal node test** | One Boolean predicate in the access policy |
| **Leaf label** | Security disposition: ALLOW / DENY / ESCALATE |
| **Root-to-leaf path** | The full evaluation trace for one input |
| **Depth $d$** | Number of conditions checked; complexity of the policy |
| **AND-tree structure** | Each clause must pass — short-circuit on first failure |
| **OR-tree structure** | Any clause grants access — single pass grants |
| **Shallow tree** | Fast evaluation; easy adversarial path-finding |
| **Deep tree** | Complex policy; more probing required to find ALLOW |
| **Test ordering at root** | Most discriminating condition first — efficiency and security |
| **Adversarial example** | Input satisfying all ALLOW path conditions |

**The Module 01 connection, completed.** In Module 01 we built truth tables for
$A \wedge T \wedge \neg S$ and identified the one row that grants access. In this
unit, that truth table row is the ALLOW leaf of the decision tree, and the attacker's
goal is to satisfy the root-to-leaf path conditions. The path conditions are the
conjunction of the feature values along the path — exactly the truth table row.

---


## Python: Visualization & Verification

**1 · ABAC Policy as a Decision Tree** — build the decision tree for
$A \wedge T \wedge \neg S$ explicitly; trace every possible input through
the tree; confirm the leaf labels match the truth table from Module 01.

**2 · Multi-Label Security Classifier** — build a four-attribute classifier
with five leaf labels (ALLOW, DENY, ESCALATE, LOG_AND_ALLOW, QUARANTINE);
visualise the full tree; show which inputs reach each leaf.

**3 · Adversarial Path Finder** — implement a black-box oracle attack that
queries the classifier to find a path to the ALLOW leaf; count the minimum
number of queries needed; compare against tree depth.


In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)
print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import itertools

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

# Leaf colours for security dispositions
LEAF_COLORS = {
    'ALLOW':         TS_GREEN,
    'DENY':          TS_RED,
    'ESCALATE':      TS_AMBER,
    'LOG_AND_ALLOW': '#1e8c8c',
    'QUARANTINE':    '#8e44ad',
}

print('Setup complete.')


### 1 · ABAC Policy Decision Tree

We build the decision tree for $A \wedge T \wedge \neg S$ exactly as described
in Section 2, trace all 8 possible inputs through the tree, and verify that
the leaf labels match the truth table from Module 01 Unit 02.


In [ ]:
# ── 1 · ABAC Policy Decision Tree ────────────────────────────────────────────

class DecisionNode:
    """A node in a decision tree."""
    def __init__(self, feature=None, label=None, left=None, right=None):
        self.feature = feature   # test feature name (None for leaves)
        self.label   = label     # leaf label (None for internal nodes)
        self.left    = left      # child when test FAILS (False)
        self.right   = right     # child when test PASSES (True)

    def is_leaf(self):
        return self.feature is None

    def classify(self, x):
        """Classify input dict x and return (label, path)."""
        path = []
        node = self
        while not node.is_leaf():
            val = x.get(node.feature, False)
            path.append((node.feature, val))
            node = node.right if val else node.left
        return node.label, path

# Build tree for A ∧ T ∧ ¬S
abac_tree = DecisionNode(
    feature='A',
    left  = DecisionNode(label='DENY'),   # A=False → DENY immediately
    right = DecisionNode(
        feature='T',
        left  = DecisionNode(label='DENY'),   # T=False → DENY
        right = DecisionNode(
            feature='S',
            left  = DecisionNode(label='ALLOW'),  # S=False (¬S=True) → ALLOW
            right = DecisionNode(label='DENY'),   # S=True  (suspended) → DENY
        )
    )
)

# Trace all 8 inputs
print("Decision tree trace for A ∧ T ∧ ¬S:")
print(f"{'Input':<30} {'Decision':>10}  Path")
print("─" * 65)

all_inputs = list(itertools.product([True, False], repeat=3))
truth_table_check = []
for A, T, S in all_inputs:
    x = {'A': A, 'T': T, 'S': S}
    label, path = abac_tree.classify(x)
    expected = 'ALLOW' if (A and T and not S) else 'DENY'
    match = '✓' if label == expected else '✗'
    path_str = ' → '.join(f"{f}={'T' if v else 'F'}" for f,v in path)
    truth_table_check.append(label == expected)
    print(f"  A={'T' if A else 'F'}, T={'T' if T else 'F'}, S={'T' if S else 'F'}"
          f"{'  ':>10} {label:>10}  {path_str}  {match}")

print(f"\nAll labels match Module 01 truth table: {all(truth_table_check)} ✓")

# ── Visualise the tree ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))

def draw_decision_tree(node, ax, x=0, y=0, dx=3.5, depth=0, parent_pos=None, edge_label=''):
    if node.is_leaf():
        color = LEAF_COLORS.get(node.label, TS_GRAY)
        rect = mpatches.FancyBboxPatch(
            (x - 0.9, y - 0.35), 1.8, 0.7,
            boxstyle='round,pad=0.1', facecolor=color,
            edgecolor='white', linewidth=2
        )
        ax.add_patch(rect)
        ax.text(x, y, node.label, ha='center', va='center',
                fontsize=8, fontweight='bold', color='white')
    else:
        circle = plt.Circle((x, y), 0.42, color=TS_BLUE, zorder=3, alpha=0.92)
        ax.add_patch(circle)
        ax.text(x, y, node.feature, ha='center', va='center',
                fontsize=10, fontweight='bold', color='white', zorder=4)

    if parent_pos:
        ax.annotate('', xy=(x, y + 0.45), xytext=(parent_pos[0], parent_pos[1] - 0.45),
                    arrowprops=dict(arrowstyle='->', color=TS_GRAY, lw=1.8))
        mx = (x + parent_pos[0]) / 2
        my = (y + parent_pos[1]) / 2
        ax.text(mx + 0.2, my, edge_label, fontsize=8, color=TS_GRAY, style='italic')

    if not node.is_leaf():
        draw_decision_tree(node.left,  ax, x - dx, y - 2.2, dx*0.55, depth+1,
                           (x, y), 'False
(F)')
        draw_decision_tree(node.right, ax, x + dx, y - 2.2, dx*0.55, depth+1,
                           (x, y), 'True
(T)')

ax.set_xlim(-9, 9); ax.set_ylim(-7.5, 1.5)
draw_decision_tree(abac_tree, ax, x=0, y=0)
ax.set_title('Decision Tree for $A \wedge T \wedge \neg S$
'
             'Left branch = test FAILS (False)  |  Right branch = test PASSES (True)',
             pad=10, fontsize=11)

legend = [mpatches.Patch(color=TS_GREEN, label='ALLOW'),
          mpatches.Patch(color=TS_RED,   label='DENY'),
          mpatches.Patch(color=TS_BLUE,  label='Test node')]
ax.legend(handles=legend, loc='lower left', fontsize=9)
ax.axis('off'); ax.set_facecolor('white'); fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()


**What do you see?**

The trace table maps exactly to the Module 01 truth table — all 8 rows, all
correct labels (✓ on every row). The short-circuit behaviour is visible in the
path column: if $A$ is False, the path is just `A=F → DENY` — $T$ and $S$ are
never tested. This is the Boolean AND short-circuit from Module 01, now visible
as a tree structure: the left branch of the root leads directly to a DENY leaf.

The tree diagram shows three levels (depth 3) with 4 leaf nodes. The single
ALLOW leaf is reached only by the path `A=T → T=T → S=F` — exactly the one
truth table row that grants access.

**The adversarial question:** given only the ability to call `classify(x)` and
observe the output, how would an attacker find the ALLOW leaf? The answer is
directly below.


### 2 · Multi-Label Security Classifier

We extend to a four-attribute policy with five leaf labels. The tree now
distinguishes between simple ALLOW, LOG_AND_ALLOW (high-privilege users),
ESCALATE (active but unverified users), QUARANTINE (suspicious patterns),
and DENY. The tree depth increases and the leaf colour map shows the distribution
of dispositions.


In [ ]:
# ── 2 · Multi-Label Security Classifier ──────────────────────────────────────
#
# Features: A (admin), T (time), S (suspended), M (MFA verified)
# Policy:
#   Suspended → always DENY
#   Not admin, not MFA → DENY
#   Admin + MFA + time → LOG_AND_ALLOW (privileged, full audit)
#   Admin + MFA + no time → ESCALATE (needs re-auth outside window)
#   Admin + no MFA → QUARANTINE (admin without MFA is suspicious)
#   Non-admin + MFA + time → ALLOW
#   Non-admin + MFA + no time → DENY

multi_tree = DecisionNode(
    feature='S',
    right = DecisionNode(label='DENY'),          # suspended → DENY
    left  = DecisionNode(
        feature='M',
        left  = DecisionNode(                    # no MFA
            feature='A',
            left  = DecisionNode(label='DENY'),  # no MFA, not admin
            right = DecisionNode(label='QUARANTINE')  # admin, no MFA
        ),
        right = DecisionNode(                    # has MFA
            feature='A',
            left  = DecisionNode(               # not admin, has MFA
                feature='T',
                left  = DecisionNode(label='DENY'),   # no MFA + outside window
                right = DecisionNode(label='ALLOW'),  # non-admin, MFA, in window
            ),
            right = DecisionNode(               # admin, has MFA
                feature='T',
                left  = DecisionNode(label='ESCALATE'),      # outside time window
                right = DecisionNode(label='LOG_AND_ALLOW'), # full privilege
            )
        )
    )
)

# Classify all 16 inputs
print("Multi-label classifier — all 16 inputs:")
print(f"{'Input':<32} {'Decision'}")
print("─" * 55)

leaf_counts = {}
for A, T, S, M in itertools.product([True, False], repeat=4):
    x = {'A': A, 'T': T, 'S': S, 'M': M}
    label, path = multi_tree.classify(x)
    leaf_counts[label] = leaf_counts.get(label, 0) + 1
    print(f"  A={'T' if A else 'F'}, T={'T' if T else 'F'}, "
          f"S={'T' if S else 'F'}, M={'T' if M else 'F'}  →  {label}")

print(f"\nDisposition distribution:")
for lbl, count in sorted(leaf_counts.items(), key=lambda x: -x[1]):
    bar = '█' * count
    print(f"  {lbl:<16} {count:>2}  {bar}")

# ── Bar chart of dispositions ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
labels_ordered = sorted(leaf_counts.items(), key=lambda x: -x[1])
names  = [l for l,_ in labels_ordered]
counts = [c for _,c in labels_ordered]
colors = [LEAF_COLORS.get(n, TS_GRAY) for n in names]

bars = ax.bar(names, counts, color=colors, edgecolor='white', width=0.6)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(count), ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('Number of inputs receiving this disposition')
ax.set_title('Multi-Label Security Classifier Disposition Distribution
'
             '4 attributes (A, T, S, M) → 16 possible inputs → 5 dispositions',
             pad=10, fontsize=11)
ax.set_ylim(0, max(counts) + 1.5)
ax.grid(axis='y', alpha=0.3)
fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()


**What do you see?**

The distribution shows that most inputs are DENY — the classifier is conservative.
The privileged LOG_AND_ALLOW path (admin + MFA + time) is the rarest disposition,
reached by only 1 input. QUARANTINE captures the dangerous case of an admin without
MFA — suspicious enough to restrict, but not immediately block.

The disposition distribution is a security property: a well-designed classifier
should have relatively few ALLOW paths and require multiple conditions to be true
simultaneously. A classifier where ALLOW is the most common outcome is almost
certainly misconfigured.

Notice the ESCALATE disposition — it handles the case where all conditions are
met except the time window. Rather than denying outright (which might block
legitimate emergency access), it asks for re-authentication. This is the
multi-class leaf label system doing real security work.


### 3 · Adversarial Path Finder

We implement a black-box oracle attack against the ABAC classifier. The attacker
starts knowing only the feature names (not the tree structure or leaf assignments).
They submit queries, observe dispositions, and use binary search-like logic to
find the path to ALLOW.


In [ ]:
# ── 3 · Adversarial Path Finder ──────────────────────────────────────────────

def oracle(x, tree=abac_tree):
    """Black-box classifier: input dict → label string."""
    label, _ = tree.classify(x)
    return label

features = ['A', 'T', 'S']

def adversarial_path_find(oracle_fn, features, target='ALLOW'):
    """
    Black-box adversarial path-finding.
    Tries all combinations of True/False for each feature systematically,
    logging each oracle query and stopping when ALLOW is found.
    This simulates a binary probe attack.
    """
    queries = []
    for combo in itertools.product([True, False], repeat=len(features)):
        x = dict(zip(features, combo))
        result = oracle_fn(x)
        queries.append((dict(x), result))
        if result == target:
            return x, queries
    return None, queries

allow_input, query_log = adversarial_path_find(oracle, features)

print(f"Adversarial path-finding against ABAC classifier")
print(f"Target: ALLOW leaf")
print(f"Oracle: black-box (tree structure unknown)")
print(f"Features known: {features}")
print()
print(f"Query log:")
for i, (x, result) in enumerate(query_log, 1):
    input_str = ', '.join(f"{k}={'T' if v else 'F'}" for k,v in x.items())
    found = '  ← TARGET FOUND' if result == 'ALLOW' else ''
    print(f"  Query {i}: {{{input_str}}}  →  {result}{found}")

print()
print(f"Queries needed:  {len(query_log)} / {2**len(features)} possible inputs")
print(f"Tree depth:      {len(features)}")
print(f"Found ALLOW at:  {allow_input}")
print()
print(f"Worst case: {2**len(features)} queries (exhaustive search)")
print(f"Best case:  {len(features)} queries (one per level, binary search)")
print(f"This run:   {len(query_log)} queries")

# ── Visualise query path ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(-9, 9); ax.set_ylim(-7.5, 1.5)

# Re-draw the ABAC tree with the found path highlighted
def draw_with_path(node, ax, path_input, x=0, y=0, dx=3.5, depth=0,
                   parent_pos=None, edge_label='', depth_tracker=[0]):
    is_on_path = path_input is not None
    color = LEAF_COLORS.get(node.label, TS_GRAY) if node.is_leaf() else TS_BLUE
    highlight = TS_AMBER if is_on_path else None

    if node.is_leaf():
        fc = TS_AMBER if is_on_path and node.label == 'ALLOW' else color
        rect = mpatches.FancyBboxPatch(
            (x - 0.9, y - 0.35), 1.8, 0.7,
            boxstyle='round,pad=0.1', facecolor=fc,
            edgecolor='white' if not is_on_path else TS_AMBER, linewidth=2.5
        )
        ax.add_patch(rect)
        ax.text(x, y, node.label, ha='center', va='center',
                fontsize=8, fontweight='bold', color='white')
    else:
        ec = TS_AMBER if is_on_path else 'white'
        lw = 3.5 if is_on_path else 0
        circle = plt.Circle((x, y), 0.42, color=TS_BLUE, zorder=3, alpha=0.92)
        ax.add_patch(circle)
        if is_on_path:
            circle2 = plt.Circle((x, y), 0.48, color=TS_AMBER, zorder=2, alpha=0.5)
            ax.add_patch(circle2)
        ax.text(x, y, node.feature, ha='center', va='center',
                fontsize=10, fontweight='bold', color='white', zorder=4)

    if parent_pos:
        lw = 3.0 if is_on_path else 1.8
        color_e = TS_AMBER if is_on_path else TS_GRAY
        ax.annotate('', xy=(x, y + 0.45), xytext=(parent_pos[0], parent_pos[1] - 0.45),
                    arrowprops=dict(arrowstyle='->', color=color_e, lw=lw))
        mx = (x + parent_pos[0]) / 2
        my = (y + parent_pos[1]) / 2
        ax.text(mx + 0.2, my, edge_label, fontsize=8, color=TS_GRAY, style='italic')

    if not node.is_leaf() and path_input is not None:
        feat_val = path_input.get(node.feature)
        left_on_path  = (feat_val == False)
        right_on_path = (feat_val == True)
        draw_with_path(node.left,  ax, path_input if left_on_path  else None,
                       x-dx, y-2.2, dx*0.55, depth+1, (x,y), 'F')
        draw_with_path(node.right, ax, path_input if right_on_path else None,
                       x+dx, y-2.2, dx*0.55, depth+1, (x,y), 'T')
    elif not node.is_leaf():
        draw_with_path(node.left,  ax, None, x-dx, y-2.2, dx*0.55, depth+1, (x,y), 'F')
        draw_with_path(node.right, ax, None, x+dx, y-2.2, dx*0.55, depth+1, (x,y), 'T')

draw_with_path(abac_tree, ax, allow_input)

ax.text(0, 1.1, f'Adversarial path found in {len(query_log)} queries  '
        f'(depth={len(features)}, worst case={2**len(features)})',
        ha='center', fontsize=10, color=TS_AMBER, fontweight='bold')
ax.set_title('Adversarial Path-Finding: Highlighted Path to ALLOW
'
             'Amber = attacker's discovered path', pad=10, fontsize=11)

legend = [mpatches.Patch(color=TS_AMBER, label='Attacker's path to ALLOW'),
          mpatches.Patch(color=TS_GREEN,  label='ALLOW leaf'),
          mpatches.Patch(color=TS_RED,    label='DENY leaf'),
          mpatches.Patch(color=TS_BLUE,   label='Test node')]
ax.legend(handles=legend, loc='lower left', fontsize=9)
ax.axis('off'); ax.set_facecolor('white'); fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()


**What do you see?**

The query log shows the attacker's systematic probe of the classifier. The
amber highlighted path shows the route from root to ALLOW leaf — the exact
sequence of conditions that must be satisfied. The attacker found it in
at most $2^n = 8$ queries (one per truth table row).

**The security implications are direct:**

1. **A decision tree classifier with $n$ Boolean features can be fully reverse-engineered
   in at most $2^n$ queries.** For $n = 3$ that is 8 queries; for $n = 10$ that is
   1,024. This is why shallow decision trees are not suitable as security gates
   against determined adversaries.

2. **The ALLOW path is a conjunction of conditions.** Once the attacker knows the
   conditions (from probing), they can craft inputs that satisfy exactly those
   conditions — a direct adversarial example.

3. **Adding more conditions (deeper tree) raises the query cost but does not
   eliminate it.** True security against a black-box attacker requires conditions
   the attacker cannot satisfy, not just conditions the attacker cannot quickly
   discover — a principle that connects directly to the CVSS-based edge weights
   from Module 05.


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. OPTIMAL TEST ORDERING
#    The order of tests in a decision tree affects average evaluation cost.
#    For the ABAC tree, suppose we know that in practice:
#      P(A=True) = 0.05  (5% of users are admin)
#      P(T=True) = 0.80  (80% of requests are in-window)
#      P(S=True) = 0.02  (2% of users are suspended)
#    Build all 6 orderings of {A, T, S} as decision trees.
#    For each ordering, compute the expected number of tests evaluated
#    per classification request (weighted by feature probabilities).
#    Which ordering minimises expected cost? Why?
#
# 2. RANDOM FOREST AS MULTIPLE TREES
#    A random forest is an ensemble of decision trees — each tree votes on
#    the classification. Implement a simple majority-vote random forest
#    with 5 decision trees, each using a random subset of 2 features
#    from {A, T, S, M}.
#    Does the ensemble change the adversarial path-finding cost?
#    (Hint: the attacker must now find an input that satisfies the majority.)
#
# 3. DECISION TREE ↔ BOOLEAN FORMULA
#    Every decision tree is equivalent to a Boolean formula (DNF).
#    Convert the multi-label tree from Section 2 into a set of DNF formulas —
#    one formula per leaf label, capturing exactly which inputs reach that leaf.
#    Verify: the DNF formulas should partition all 16 inputs with no overlap.
#    Compare to the truth tables from Module 01 — are they the same structure?

# Your work here:


---

## Summary

| Concept | Definition | Security meaning |
|---|---|---|
| **Decision tree** | Binary tree: internal nodes = tests, leaves = labels | Policy evaluation with explicit trace |
| **Root-to-leaf path** | Unique path for any input | Full evaluation trace — audit log in tree form |
| **Leaf label** | Security disposition | ALLOW / DENY / ESCALATE / LOG / QUARANTINE |
| **Tree depth** | Height — number of conditions checked | Policy complexity; adversarial search cost |
| **AND-tree** | Short-circuit on first False | Efficient denial; first failure stops evaluation |
| **Adversarial path** | Input satisfying all ALLOW path conditions | The adversarial example, in tree form |
| **Oracle attack** | Black-box probing to find ALLOW | $2^n$ worst-case queries for $n$ Boolean features |
| **Test ordering** | Which condition at root | Efficiency + information-leakage tradeoff |

---

## Module 06 Complete

| Unit | Content | Payoff |
|---|---|---|
| **01** | Tree definitions, properties, traversals, spanning trees | The vocabulary and structure of hierarchical data |
| **02** | Parse trees, grammars, prompt injection | Injection = user token at non-terminal position |
| **03** | Decision trees, classifiers, adversarial path-finding | Policy as tree; attacker finds path to ALLOW |

The three units form one argument: **trees are the formal structure of hierarchical
decisions — whether that decision is how to parse an expression, how to interpret a
prompt, or whether to grant access. Understanding the tree is understanding the decision.**

---

## Up Next

**Module 07 — Combinatorics and Counting**

We shift from structure to enumeration. Combinatorics answers the question:
*how many* — how many possible inputs exist, how many attack paths are there,
how many adversarial examples fit within a given constraint. The counting
principles of Module 07 are the foundation of attack surface analysis and
the adversarial enumeration thread that runs through the whole course.

$\rightarrow$ `modules/module-07/unit-01-counting-principles.ipynb`
